In [1]:
import random
import json
import numpy as np

In [2]:
# ---- Class names ----
CLASS_NAMES = {
    0: "early",
    1: "on_time",
    2: "delay"
}

# ---- Feature columns ----
FEATURE_COLUMNS = [
    'profit_per_order', 'order_item_discount', 'order_item_product_price',
    'order_item_profit_ratio', 'order_item_quantity', 'sales',
    'order_profit_per_order', 'shipping_mode', 'distance_normalized',
    'order_to_shipment_days', 'order_shipping_time',
    'order_to_shipment_planned_days', 'shipment_delay_days',
    'performance_score_order_full_location',
    'performance_score_customer_full_location',
    'performance_score_order_dayofweek',
    'performance_score_shipping_dayofweek', 'performance_score_order_hour',
    'performance_score_shipping_hour', 'performance_score_order_daynight',
    'performance_score_ship_daynight', 'payment_type_CASH',
    'payment_type_DEBIT', 'payment_type_PAYMENT', 'payment_type_TRANSFER'
]

# ---- Parameters ----
NUM_SAMPLES = 1000
TOP_FEATURES = 2  # Number of positive/negative features to pick per sample

# ---- Function to generate a random explanation dict ----
def generate_random_explanation(sample_idx):
    # Random predicted probabilities that sum to 1
    probs = np.random.dirichlet(np.ones(len(CLASS_NAMES)), size=1)[0]
    
    pred_class_idx = int(np.argmax(probs))
    pred_label = CLASS_NAMES[pred_class_idx]

    # Pick random positive features with random SHAP values
    top_positive = []
    top_negative = []
    pos_features = random.sample(FEATURE_COLUMNS, TOP_FEATURES)
    neg_features = random.sample([f for f in FEATURE_COLUMNS if f not in pos_features], TOP_FEATURES)
    
    for f in pos_features:
        top_positive.append({f: round(random.uniform(0.01, 0.2), 4)})
    for f in neg_features:
        top_negative.append({f: round(random.uniform(-0.2, -0.01), 4)})

    # Build predicted probabilities dictionary
    prob_dict = {CLASS_NAMES[i]: float(round(probs[i], 4)) for i in range(len(CLASS_NAMES))}

    explanation = {
        "sample_index": sample_idx,
        "predicted_class": pred_label,
        "predicted_probabilities": prob_dict,
        "top_positive_features": top_positive,
        "top_negative_features": top_negative
    }

    return explanation

# ---- Generate all samples ----
all_explanations = [generate_random_explanation(i) for i in range(NUM_SAMPLES)]

# ---- Save to file ----
with open("simulated_shap_explanations.json", "w") as f:
    json.dump(all_explanations, f, indent=2)

print(f"{NUM_SAMPLES} simulated SHAP explanations saved to 'simulated_shap_explanations.json'")

1000 simulated SHAP explanations saved to 'simulated_shap_explanations.json'


In [3]:
import json
import random

# ---- Load your 1000 samples JSON ----
with open("simulated_shap_explanations.json", "r") as f:
    all_samples = json.load(f)

# ---- Feature meaning mapping ----
FEATURE_MEANINGS = {
    'profit_per_order': "profit per order",
    'order_item_discount': "discount given on the order item",
    'order_item_product_price': "price of the order item",
    'order_item_profit_ratio': "profit ratio for the order item",
    'order_item_quantity': "quantity of items in the order",
    'sales': "total sales for the order",
    'order_profit_per_order': "profit generated from the order",
    'shipping_mode': "shipping method selected",
    'distance_normalized': "distance between store and delivery location",
    'order_to_shipment_days': "days between order and shipment",
    'order_shipping_time': "time taken to ship the order",
    'order_to_shipment_planned_days': "planned shipping days after order",
    'shipment_delay_days': "delay in shipment from planned date",
    'performance_score_order_full_location': "impact of order's location",
    'performance_score_customer_full_location': "impact of customer's location",
    'performance_score_order_dayofweek': "impact of order day of week",
    'performance_score_shipping_dayofweek': "impact of shipping day of week",
    'performance_score_order_hour': "impact of order hour",
    'performance_score_shipping_hour': "impact of shipping hour",
    'performance_score_order_daynight': "impact of order time of the day",
    'performance_score_ship_daynight': "impact of shipping time of the day",
    'payment_type_CASH': "payment via cash",
    'payment_type_DEBIT': "payment via debit card",
    'payment_type_PAYMENT': "payment via online payment",
    'payment_type_TRANSFER': "payment via bank transfer"
}

In [ ]:
import random

# -----------------------------
# Templates
# -----------------------------
PRED_TEMPLATES = [
    "The shipment is expected to be {class_label} with a probability of {prob:.0%}, indicating the model's primary prediction.",
    "Our model predicts the shipment will be {class_label} ({prob:.0%} likelihood), reflecting operational trends.",
    "Forecast indicates {class_label} as the likely outcome, with a probability of {prob:.0%}, based on historical data.",
    "Analysis shows a {prob:.0%} chance that this shipment will be {class_label}, suggesting strong model confidence.",
    "We anticipate the delivery status to be {class_label} ({prob:.0%}), highlighting expected performance patterns."
]

POS_TEMPLATES = [
    "Positive contributing factors include {features}, which increased the likelihood of {class_label}.",
    "{features} strongly supported the predicted outcome, boosting the probability of {class_label}.",
    "Key drivers like {features} reinforced the model's prediction, making {class_label} more probable.",
    "Factors such as {features} had a beneficial effect, enhancing the chances of {class_label}.",
    "The model attributes a higher likelihood of {class_label} to positive influences, including {features}."
]

NEG_TEMPLATES = [
    "However, {features} had a slight negative impact, slightly reducing the chance of {class_label}.",
    "Factors such as {features} negatively affected the predicted outcome, lowering the probability of {class_label}.",
    "On the downside, {features} decreased the likelihood of {class_label} a little.",
    "Negative influences like {features} slightly reduced the probability of the shipment being {class_label}.",
    "Although {features} detracted slightly, {class_label} remains the most likely outcome."
]

OTHER_CLASSES_TEMPLATES = [
    "Other possible outcomes include {other_probs}, showing lower chances compared to the predicted class ({pred_prob:.0%}).",
    "Besides the predicted class, alternative probabilities are: {other_probs}, all lower than {class_label} ({pred_prob:.0%}).",
    "The shipment could also be {other_probs}, though {class_label} ({pred_prob:.0%}) remains the main forecast.",
    "Additional outcomes include {other_probs}, reinforcing that {class_label} ({pred_prob:.0%}) is the strongest prediction.",
    "While other classes such as {other_probs} exist, {class_label} ({pred_prob:.0%}) dominates as the expected outcome."
]


def generate_explanation(sample_json, feature_meanings, num_variations=10):
    """
    Generate multiple 4-line explanations (num_variations) for a single sample,
    using feature-specific contributions for positive/negative impacts.
    """
    class_label = sample_json['predicted_class']
    pred_prob = sample_json['predicted_probabilities'][class_label]

    # Map positive features and their SHAP impacts in percentage
    pos_features_list = []
    for f in sample_json['top_positive_features']:
        feat_name = list(f.keys())[0]
        feat_meaning = feature_meanings.get(feat_name, feat_name)
        feat_value = f[feat_name] * 100  # convert to percentage
        pos_features_list.append(f"{feat_meaning} (+{feat_value:.1f}%)")  # show positive impact

    pos_features = ", ".join(pos_features_list)

    # Map negative features and their SHAP impacts in percentage
    neg_features_list = []
    for f in sample_json['top_negative_features']:
        feat_name = list(f.keys())[0]
        feat_meaning = feature_meanings.get(feat_name, feat_name)
        feat_value = f[feat_name] * 100  # convert to percentage
        neg_features_list.append(f"{feat_meaning} ({feat_value:.1f}%)")  # show negative impact

    neg_features = ", ".join(neg_features_list)

    # Other classes probabilities
    other_classes = [cls for cls in sample_json['predicted_probabilities'] if cls != class_label]
    other_probs_str = ", ".join([f"{cls} ({sample_json['predicted_probabilities'][cls]:.0%})"
                                 for cls in other_classes])

    explanations = []
    for _ in range(num_variations):
        pred_sentence = random.choice(PRED_TEMPLATES).format(class_label=class_label, prob=pred_prob)
        pos_sentence = random.choice(POS_TEMPLATES).format(features=pos_features, class_label=class_label)
        neg_sentence = random.choice(NEG_TEMPLATES).format(features=neg_features, class_label=class_label)
        other_sentence = random.choice(OTHER_CLASSES_TEMPLATES).format(other_probs=other_probs_str, class_label=class_label, pred_prob=pred_prob)

        full_explanation = f"{pred_sentence} {pos_sentence} {neg_sentence} {other_sentence}"
        explanations.append(full_explanation)

    return explanations


# ---- Generate explanations for all samples ----
all_explanations = []
for sample in all_samples:
    exps = generate_explanation(sample, FEATURE_MEANINGS, num_variations=10)
    all_explanations.append({
        "input": {
            "predicted_class": sample['predicted_class'],
            "predicted_probabilities": sample['predicted_probabilities'],
            "top_positive_features": sample['top_positive_features'],
            "top_negative_features": sample['top_negative_features']
        },
        "output": exps
    })

# ---- Save explanations to JSON file ----
with open("synthetic_nl_explanations.json", "w") as f:
    json.dump(all_explanations, f, indent=2)

print("Synthetic explanations generated for all samples and saved to 'synthetic_nl_explanations.json'")

Synthetic explanations generated for all samples and saved to 'synthetic_nl_explanations.json'


: 